In [0]:
%run "../00_config/00_config"

In [0]:
%run "../../src/utilities/logger"

In [0]:
from delta.tables import DeltaTable

dbutils.widgets.text("start_date", "2026-01-01", "Start Ingestion Date")
dbutils.widgets.text("end_date", datetime.date.today().strftime("%Y-%m-%d"), "End Ingestion Date")

start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
secret_salt = get_secret_salt()

def process_bronze_to_silver():
    # Read Bronze dataset for date range
    bronze_df = (spark.table(TABLE_BRONZE_FULL)
                 .filter((col("ingestion_date") >= start_date) & (col("ingestion_date") <= end_date)))

    # Step 4.2 & 4.5: Flatten, Cast, and Protect PII
    flattened_df = bronze_df.select(
        col("itemId").alias("item_id"),
        # Mask phone numbers and emails in titles
        regexp_replace(col("title"), r"(\d{3}-\d{3}-\d{4}|\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b)", "[REDACTED]").alias("title"),
        col("categories")[0]["categoryId"].alias("category_id"),
        col("categories")[0]["categoryName"].alias("category_name"),
        # Pseudonymize seller username with HMAC SHA-256
        sha2(concat(col("seller.username"), lit(secret_salt)), 256).alias("seller_key"),
        col("seller.feedbackScore").cast("int").alias("feedback_score"),
        col("seller.feedbackPercentage").cast("double").alias("feedback_percentage"),
        col("price.value").cast("decimal(18,2)").alias("price"),
        col("price.currency").alias("currency"),
        col("condition"),
        to_timestamp(col("itemCreationDate")).alias("listing_date"),
        col("itemLocation.country").alias("country"),
        col("itemLocation.stateOrProvince").alias("state"),
        col("itemLocation.city").alias("city"),
        col("batch_id"),
        col("ingestion_timestamp"),
        col("_rescued_data")
    )

    # Step 4.4: Quality Rules & Quarantine Logic
    valid_cond = (
        col("item_id").isNotNull() & 
        col("price").isNotNull() & 
        (col("price") > 0) & 
        col("_rescued_data").isNull()
    )

    quarantined_df = flattened_df.filter(~valid_cond).withColumn(
        "reason_code", 
        expr("""CASE 
            WHEN item_id IS NULL THEN 'MISSING_ITEM_ID'
            WHEN price IS NULL OR price <= 0 THEN 'INVALID_PRICE'
            WHEN _rescued_data IS NOT NULL THEN 'SCHEMA_DRIFT_DETECTED'
            ELSE 'UNKNOWN_VALIDATION_FAILURE' END""")
    ).withColumn("load_timestamp", current_timestamp())

    # Write failures to silver_quarantine
    quarantined_count = quarantined_df.count()
    if quarantined_count > 0:
        quarantined_df.write.format("delta").mode("append").saveAsTable(TABLE_SILVER_QUARANTINE)

    valid_df = flattened_df.filter(valid_cond)

    # Step 4.6: Deduplicate within batch
    window_spec = Window.partitionBy("item_id").orderBy(col("ingestion_timestamp").desc())
    deduped_df = (valid_df
                  .withColumn("rn", row_number().over(window_spec))
                  .filter("rn = 1")
                  .drop("rn")
                  .withColumn("row_hash", sha2(concat(col("price"), col("condition"), col("title")), 256))
                  .withColumn("load_timestamp", current_timestamp()))

    # Step 4.7: MERGE INTO silver_listings
    if not spark.catalog.tableExists(TABLE_SILVER_LISTINGS):
        deduped_df.write.format("delta").saveAsTable(TABLE_SILVER_LISTINGS)
        inserted, updated = deduped_df.count(), 0
    else:
        target_table = DeltaTable.forName(spark, TABLE_SILVER_LISTINGS)
        target_table.alias("t").merge(
            deduped_df.alias("s"),
            "t.item_id = s.item_id"
        ).whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={
                "title": "s.title",
                "price": "s.price",
                "condition": "s.condition",
                "row_hash": "s.row_hash",
                "load_timestamp": "s.load_timestamp"
            }
        ).whenNotMatchedInsertAll().execute()

        inserted, updated = get_last_merge_metrics(spark, TABLE_SILVER_LISTINGS)

    return inserted, updated, quarantined_count

run_step(spark, TABLE_LOGS, "Bronze-to-Silver", "INCREMENTAL", TABLE_SILVER_LISTINGS, f"{start_date} to {end_date}", process_bronze_to_silver)

In [0]:
%sql
-- Query 1: Verify Duplicate Primary Keys (Must return 0 rows)
SELECT item_id, COUNT(*) 
FROM workspace.ebay_pipeline.silver_listings 
GROUP BY item_id 
HAVING COUNT(*) > 1;

-- Query 2: Verify Audit Log Metrics (Second run must show 0 inserted, 0 updated)
SELECT layer, target_table, status, rows_inserted, rows_updated, start_time 
FROM workspace.ebay_pipeline.pipeline_execution_logs 
ORDER BY start_time DESC LIMIT 2;